This table creates and replaces Microsoft SQL tables: FL_DB.DBO.ADVORDERS_CURRENTSTATUS and QS_Query.DBO.ADVORDERS_CURRENTSTATUS

Step 1: Bring in each table with the needed columns separately and save as its own dataframe. This will allow for error checking at the table level. 

In [0]:
from pyspark.sql import functions as F

#Bring in needed columns from FSS19_contract_master 
#Bring in Needed fields and create df
#Needed fields: "uei", "legal_bus_name", "arn_aro_dys", "sched_no", "ship_del_cd", "portfolio", "pco_name", "byr_name"
#Join fields: "gsam_cont_no"- also needed
#restrict to sched_no = "MAS"

fss19_contBase = "/Volumes/fas_eda_fss19_cmf_prd/gold/fss19_contract_master"

folders = dbutils.fs.ls(fss19_contBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"FSS19 Contract Latest folder: {latest_folder}")

# load only the latest folder, apply CMF business rules
fss19_AdvOrder = (
    spark.read.parquet(latest_folder)
    .filter(F.col("cont_ind") == "F")
    .select(
        F.col("gsam_cont_no"),
        F.col("contr_name"),
        F.col("sched_no"),
        F.col("uei"),
        F.col("legal_bus_name"),
        F.col("arn_aro_dys"),
        F.col("ship_del_cd"),
        F.col("portfolio"),
        F.col("pco_name"),
        F.col("byr_name"),
        F.col("e_mail_adrs")
    )        
    )

print (f"Number of records in FSS19 Contract: {fss19_AdvOrder.count()}")
display(f"distinct values for ship_del_cd : {fss19_AdvOrder.select(F.col('ship_del_cd')).distinct()}")
fss19_AdvOrder.select(F.col('ship_del_cd')).distinct().show()
display(f"distinct values for portfolio: {fss19_AdvOrder.select('portfolio').distinct()}")
fss19_AdvOrder.select('portfolio').distinct().show()
display(f"distinct values for sched_no : {fss19_AdvOrder.select('sched_no').distinct()}")
fss19_AdvOrder.select('sched_no').distinct().show()
display(f"distinct values for arn_aro_dys : {fss19_AdvOrder.select('arn_aro_dys').distinct()}")
fss19_AdvOrder.select('arn_aro_dys').distinct().show()
display(fss19_AdvOrder.limit(10))

#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/fss19_AdvOrder.csv"
fss19_AdvOrder.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/fss19_AdvOrder.csv")

In [0]:
from pyspark.sql import functions as F
#Bring in Needed fields and create df
#Needed fields:"is_edi", 
#check these fields: is_ebuy_po
# bv_order_identity
# purchase_identityX
# purchase_numberX
# is_po X
# is_requisition
# is_ediX
# is_edd_ng
# is_oversea
# is_4pl
# is_adv_po X
# is_edd
#Join fields: "purchase_number"-with advantage_portal_order_assn , "extract_day_key" -with advantage_item_option "purchase_day_key"- not with any selected tables, "purchase_key"-  "purchase_identity"-with advantage_portal_order_assn
adv_purchase_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_purchase/*/*.parquet"
# adv_purchase_df = spark.read.parquet(adv_purchase_path)
#(F.col("is_adv_po") == 1) just looking at MAS 

adv_purchaseBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_purchase"
folders = dbutils.fs.ls(adv_purchaseBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Purchase Latest folder: {latest_folder}")

adv_purchase = (
    spark.read.parquet(latest_folder)
    .filter(F.col("is_po") == 1)
    .filter(F.col("is_adv_po") == 1)
    .select(
        F.col("purchase_number"),
        F.col("shipping_identity"), 
        F.col("purchase_identity"),
        F.col("bv_order_identity"),
        F.col("is_requisition"),
        F.col("is_edi"),
        F.col("is_edd_ng"),
        F.col("is_oversea"),
        F.col("is_4pl"),
        F.col("is_edd"),
        F.col("is_ebuy_po"),
        F.col("is_po"),
        F.col("is_adv_po"),
        F.col("purchase_hour"), 
        F.col("purchase_minute"),
        F.col("purchase_day_key"),
        F.col("purchase_key"),
        F.col("extract_day_key")
    )
)

display(adv_purchase.limit(10))
print(f"adv_purchase count: {adv_purchase.count()}")
adv_purchase.select("is_edi").distinct().show()
print(f"count when edi=1: {adv_purchase.filter(F.col('is_edi') == 1).count()}")


#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase.csv"
adv_purchase.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase.csv")


In [0]:
from pyspark.sql import functions as F

#Bring in adv_purchase_df
#advantage_purchase
adv_purchase_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase.csv"
adv_purchase_df = spark.read.csv(adv_purchase_csvpath, header=True)

print(f"adv_purchase_df count: {adv_purchase_df.count()}")

#create df from advantage_day
adv_day_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_day"
folders = dbutils.fs.ls(adv_day_path)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Purchase Latest folder: {latest_folder}")

adv_day = (
    spark.read.parquet(latest_folder)
    .withColumn("day_date", F.try_to_date(F.col("day_date"), "yyyyDDD"))
    .select(
        F.col("day_key"),
        F.col("day_date"),
        F.col("calendar_year"),
        F.col("fiscal_year"), 
        F.col("fiscal_month")
        
    )
)
display(adv_day.limit(10))
print(f"adv_day count: {adv_day.count()}")

#save adv_day to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_day.csv"
adv_day.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_day.csv")

#Join adv_purchase with adv_day
adv_purchase_day = adv_purchase_df.join(adv_day, adv_purchase_df.purchase_day_key == adv_day.day_key, "left").drop(adv_day.day_key)
# display(adv_purchase_day.limit(10))
print(f"adv_purchase_day count: {adv_purchase_day.count()}")

#rename columns to order_date and order_date_key
adv_purchase_day = adv_purchase_day.withColumnRenamed("day_date", "order_date").withColumnRenamed("calendar_year", "order_calendar_year").withColumnRenamed("fiscal_year", "order_fiscal_year").withColumnRenamed("fiscal_month", "order_fiscal_month")
display(adv_purchase_day.limit(10))

#save df to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase_day.csv"
adv_purchase_day.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase_day.csv")

In [0]:
from pyspark.sql import functions as F

#Bring in Needed fields and create df
#Needed fields: "quantity", "line_status", on advantage_portal_order_assn too: "est_ship_date", "tracking_num", "status", "status_date", "display_flag"-see what this is
#Join fields: maybe "line_item_identity", "purchase_number", "contract_num" - with advantage_portal_order_assn 

#Do not bring in id numbers that are not being used, because they differ and create more rows for a purchase_number. But they don't bring in any new information
adv_orderStat_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_order_status/*/*.parquet"
# adv_orderStat_df = spark.read.parquet(adv_orderStat_path)

adv_orderStatBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_order_status"
folders = dbutils.fs.ls(adv_orderStatBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Order Status Latest folder: {latest_folder}")
# adv_line_item_prop_df = spark.read.parquet(latest_folder)

adv_orderStat =(
    spark.read.parquet(latest_folder)
     .withColumn("gsam_cont_no", F.trim(F.regexp_replace("contract_num", "-", "")))
     .withColumn("status_date", F.try_to_date(F.col("status_date"), "yyyyDDD"))
     .filter(F.col("gsam_cont_no") != "GSA")
    .select(
        F.col("purchase_number"),
        F.col("date_created"),
        F.col("line_status"),
        F.col("line_item_identity"),
        F.col("line_item_number"),
        F.col("mode"),
        F.col("mode_url"),
        F.col("tracking_num"),
        F.col("display_flag"),
        F.col("tcngbl"),
        F.col("status_date").alias("line_status_date"),
        F.col("quantity"),
        F.col("gsam_cont_no")
    )

)

display(adv_orderStat.limit(10))
print(f"adv_orderStat count: {adv_orderStat.count()}")
print(f"number of distinct est_ship_date: {adv_orderStat.select('est_ship_date').distinct().count()}")
adv_orderStat.select("est_ship_date").distinct().show()




#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderStat.csv"
adv_orderStat.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderStat.csv")


In [0]:
#Join Advantage_order_status and Fss19_cmf to restrict to only MAS contracts and their orders. 
#fss19_contract master data
fss19_cont_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/fss19_AdvOrder.csv"
fss19_cont_df = spark.read.csv(fss19_cont_path, header=True)

#advantage_order_status
adv_orderstat_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderStat.csv"
adv_orderstat_df = spark.read.csv(adv_orderstat_csvpath, header=True)

#Do an inner join on gsam_cont_no and drop duplicate column to restrict to just MAS contracts
adv_orderstat_fss19_df = fss19_cont_df.join(adv_orderstat_df, fss19_cont_df.gsam_cont_no == adv_orderstat_df.gsam_cont_no, how="inner").drop(adv_orderstat_df['gsam_cont_no'])
display(adv_orderstat_fss19_df.limit(10))
print(f"adv_orderstat_fss19_df count: {adv_orderstat_fss19_df.count()}")

#save df to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderstat_fss19_df.csv"
adv_orderstat_fss19_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderstat_fss19_df.csv")



In [0]:
%skip
from pyspark.sql import functions as F
# misses EDI do not use this table
#Bring in Needed fields and create df
#Needed fields: "status", "ship_via", "order_date", "cancel_request_date"  On advantage_order_status too: "tracking_number", "status_date"
#Join fields: "purchase_number"- with advantage_purchase,  "contract_num"

#Had to change the way this data table is ingested: because the last folder is the tmp file  and correct for :
#The error still occurs — now pointing to a different old file (2023-03-19) inside latest_folder. This means the latest snapshot folder contains historical data partitioned by date internally, and old files within it have date/timestamp columns stored as BYTE_ARRAY (string)

adv_portal_order_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_portal_order_assn/*/*.parquet"
# adv_portal_order_df = spark.read.parquet(adv_portal_order_path)

adv_portal_orderBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_portal_order_assn"
folders = dbutils.fs.ls(adv_portal_orderBase)
latest_folder = sorted([f.path for f in folders if f.name[:4].isdigit()], reverse=True)[0]
print(f"ADV Portal Order Latest folder: {latest_folder}")

adv_portal_order = (
    spark.read.parquet(latest_folder)
    .withColumn("gsam_cont_no", F.trim(F.regexp_replace("contract_num", "-", "")))
    .select(
        F.col("purchase_number"),
        F.col("status"),
        F.col("status_date"),
        F.col("ship_via"),
        F.col("purchase_identity"),
        F.col("is_active"), 
        F.col("order_date"),
        F.col("last_mod_date"),
        F.col("tracking_number"), 
        F.col("cancel_request_date"),
        F.col("gsam_cont_no")
    )
)

display(adv_portal_order.limit(10))
print(f"adv_portal_order count: {adv_portal_order.count()}")



#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/"

adv_portal_order.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_portal_order.csv")
# adv_portal_order.write.mode("overwrite").parquet("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_portal_order.parquet")

In [0]:
from pyspark.sql import functions as F

#Bring in Needed fields and create df
#Needed fields: "schedule_price" =  "unit_price"-only need "unit_price", "extended_price" Also on Order_status: "quantity", "line_item_key"
#Join fields: "shopper_key"- with advantage_shopper , "contract_key", "product_key"- with advantage_product "schedule_key" - with advantage_schedule, "purchase_key"- with advantage_purchase

advantage_line_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_line_item/*/*.parquet"
# adv_line_df = spark.read.parquet(advantage_line_path)

adv_line_itemBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_line_item"
folders = dbutils.fs.ls(adv_line_itemBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Line Item Latest folder: {latest_folder}")

adv_line = (
    spark.read.parquet(latest_folder)
    .select(
        F.col("unit_price"),
        F.col("extended_price"),
        F.col("quantity"),
        F.col("contract_key"),
        F.col("product_key"),
        F.col("purchase_key"), 
        F.col("line_item_key"),
        F.col("line_item_identity"),
        F.col("schedule_key"), 
        F.col("vendor_key"),
        F.col("shopper_key")
    )
)

display(adv_line.limit(10))
print(f"adv_line count: {adv_line.count()}")



#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_line.csv"
adv_line.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_line.csv")


In [0]:
from pyspark.sql import functions as F

#Bring in Needed fields and create df
#Needed fields:"vend_part_number", "mfr_part_number", "item_name", "gsin", "product_desc" "fob_code"-maybe this is redundant with ship_del_cd : investigate these fields: "fob_code", "delivery_code", "del_days1", "del_days2" to compare with FSS19 and advantage_contract
#Join fields: "mfr_part_number" should join with "mfr_part" from advantage_item_option, "product_key" - with advantage_line_item

advantage_product_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_product/*/*.parquet"
# adv_product_df = spark.read.parquet(advantage_product_path)

adv_productBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_product"
folders = dbutils.fs.ls(adv_productBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Product Latest folder: {latest_folder}")

adv_product = (
    spark.read.parquet(latest_folder)
    .select(
        F.col("vend_part_number"),
        F.col("mfr_part_number"),
        F.col("item_name"),
        F.col("gsin"),
        F.col("product_desc"),
        F.col("fob_code"), 
        F.col("delivery_code"), 
        F.col("del_days1"), 
        F.col("del_days2"),
        F.col("product_key")
       
    )
)

display(adv_product.limit(10))
print(f"adv_product count: {adv_product.count()}")

#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_product.csv"
adv_product.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_product.csv")

In [0]:
%skip
# from pyspark.sql import functions as F
#this is not currently needed
#Bring in Needed fields and create df
#Needed fields:"description"
#Join fields: "mfr_part"- with advantage_product




#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/"

In [0]:
from pyspark.sql import functions as F

#Bring in Needed fields and create df
#Needed fields: "sin" "subcat"-not using isn't filled out for current SINs - look for sin name or something better filled, try: "large_category", "sin_group_title", "sin_desc2"
#Join fields: "schedule_key" - with advantage_line_item
adv_schedule_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_schedule/*/*.parquet"
# adv_schedule_df = spark.read.parquet(adv_schedule_path)

adv_scheduleBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_schedule"
folders = dbutils.fs.ls(adv_scheduleBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Schedule Latest folder: {latest_folder}")

adv_schedule = (
    spark.read.parquet(latest_folder)
    .filter(F.col("schedule_number") == "MAS")
    .select( 
        F.col("schedule_number"),
        F.col("sin"),
        F.col("subcat"),
        F.col("large_category"),
        F.col("sin_group_title"),
        F.col("schedule_key")
    )
)

display(adv_schedule.limit(10))
print(f"adv_schedule count: {adv_schedule.count()}")


#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_schedule.csv"
adv_schedule.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_schedule.csv")

In [0]:
from pyspark.sql import functions as F
#Bring in Needed fields and create df
#Needed fields: "agency_name"
#Join fields: "shopper_key"- with advantage_line_item

# possible other fields: F.col("username"),   F.col("email"), F.col("bureau_name"),

advantage_shopper_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_shopper/*/*.parquet"
# adv_shopper_df = spark.read.parquet(advantage_shopper_path)

adv_shopperBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_shopper"
folders = dbutils.fs.ls(adv_shopperBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Shopper Latest folder: {latest_folder}")

adv_shopper = (
    spark.read.parquet(latest_folder)
    .select(
        F.col("agency_name"),
        F.col("shopper_key")
    )
)

display(adv_shopper.limit(10))
print(f"adv_shopper count: {adv_shopper.count()}")

#check distinct Shopper_key
distinct_shopper_key = adv_shopper.select(F.col("shopper_key")).distinct()
print(f"distinct_shopper_key count: {distinct_shopper_key.count()}")

#check distinct Agency_name
distinct_agency_name = adv_shopper.select(F.col("agency_name")).distinct()
print(f"distinct_agency_name count: {distinct_agency_name.count()}")



#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_shopper.csv"
adv_shopper.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_shopper.csv")

In [0]:
from pyspark.sql import functions as F
#may not need this table in the end, because already restricted to just mas purchase_numbers and all other information comes in from FSS19. 

#Bring in Needed fields and create df
#Needed fields: "is_gsa_schedule", "contract_number" : Bring in to compare with FSS19 and advantage_product: "fob_code", "fob", "delivery_days1", "delivery_days2", "delivery_code"
#Join fields: "contract_key"- with advantage_line_item

advantage_contract_path = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_contract/*/*.parquet"
# adv_contract_df = spark.read.parquet(advantage_contract_path)

adv_contractBase = "/Volumes/fas_eda_advantage_sales_transactions_prd/gold/advantage_contract"
folders = dbutils.fs.ls(adv_contractBase)
latest_folder = sorted([f.path for f in folders], reverse=True)[0]
print(f"ADV Shopper Latest folder: {latest_folder}")

adv_contract = (
    spark.read.parquet(latest_folder)
    .withColumn("gsam_cont_no", F.trim(F.regexp_replace("contract_number", "-", "")))
    .filter(F.col("is_gsa_schedule") == 1)
    .select(
        F.col("gsam_cont_no"),
        F.col("is_gsa_schedule"),
        F.col("contract_key"), 
        F.col("fob_code").alias("con_fob_code"), 
        F.col("fob").alias("con_fob"), 
        F.col("delivery_code").alias("con_delivery_code"), 
        F.col("delivery_days1").alias("con_delivery_days1"), 
        F.col("delivery_days2").alias("con_delivery_days2")
    )
)

display(adv_contract.limit(10))
print(f"adv_contract count: {adv_contract.count()}")



#save dfs to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_contract.csv"
adv_contract.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_contract.csv")


Step 2: Join all tables together in one new data frame. 
Order of Operations: 1. Capture all orders using order dependent tables and restrict to only MAS contracts. 
                    2.Join with non-order dependent tables
Join Advantage tables together by their join fields, and use "gsam_cont_no" to FSS19_contractmaster

Needed Logic: 
Fields that must have a value: 
Fields that should be unique: 
Fields that have a logic with another field: 

In [0]:
%skip

# First join is to FSS19 data to restrict to only MAS contracts
# join all order level tables to restrict data to only MAS orders

#Import sql functions for getting earliest and latest dates
from pyspark.sql import functions as F 
from pyspark.sql.functions import asc, desc

#fss19_contract master data
fss19_cont_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/fss19_AdvOrder.csv"
fss19_cont_df = spark.read.csv(fss19_cont_path, header=True)

#Order dependent adv tables: advantage_line_item, advantage_purchase, advantage_order_status,advantage_portal_order_assn


#advantage_portal_order_assn
adv_portal_order_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_portal_order.csv"
adv_portal_order_df = spark.read.csv(adv_portal_order_csvpath, header=True)
#Join fields: "purchase_number"- with advantage_purchase,  "contract_num"

#advantage_purchase
adv_purchase_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase.csv"
adv_purchase_df = spark.read.csv(adv_purchase_csvpath, header=True)

#Join fields: "purchase_number"-with advantage_portal_order_assn , "extract_day_key" -with advantage_item_option "purchase_day_key"- not with any selected tables

#advantage_order_status
adv_orderstat_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderStat.csv"
adv_orderstat_df = spark.read.csv(adv_orderstat_csvpath, header=True)
#Join fields: maybe "line_item_identity", "purchase_number", "contract_num" - with advantage_portal_order_assn 

#advantage_line_item
adv_line_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_line.csv"
adv_line_df = spark.read.csv(adv_line_csvpath, header=True)

#Join fields: "shopper_key"- with advantage_shopper , "contract_key", "product_key"- with advantage_product "schedule_key" - with advantage_schedule

#first join: fss19_cont_df with #advantage_portal_order_assn
order_level_1_df = fss19_cont_df.join(adv_portal_order_df, fss19_cont_df.gsam_cont_no == adv_portal_order_df.gsam_cont_no, how="inner").drop(adv_portal_order_df['gsam_cont_no'])
display(order_level_1_df.limit(10))
print(f"order_level_1_df count: {order_level_1_df.count()}")

#Check for duplicates in order_level_1_df
duplicate_rows1_count = order_level_1_df.groupBy(order_level_1_df.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows in order_level_1_df: {duplicate_rows1_count}")


#second join order_level_1_df with adv_orderstat_df
order_level_2_df = order_level_1_df.join(adv_orderstat_df, order_level_1_df.purchase_number == adv_orderstat_df.purchase_number, how="inner").drop(adv_orderstat_df['purchase_number']).drop(adv_orderstat_df['gsam_cont_no'])
display(order_level_2_df.limit(10))
print(f"order_level_2_df count: {order_level_2_df.count()}")

#Check for duplicates in order_level_2_df
duplicate_rows2_count = order_level_2_df.groupBy(order_level_2_df.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows in order_level_2_df: {duplicate_rows2_count}")


#third join adv_purchase_df on purchase_number
order_level_3_df = order_level_2_df.join(adv_purchase_df, order_level_2_df.purchase_number == adv_purchase_df.purchase_number, how="inner").drop(adv_purchase_df['purchase_number']).drop(adv_purchase_df['purchase_identity'])
display(order_level_3_df.limit(10))
print(f"order_level_3_df count: {order_level_3_df.count()}")

#Check for duplicates in order_level_3_df
duplicate_rows3_count = order_level_3_df.groupBy(order_level_3_df.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows in order_level_3_df: {duplicate_rows3_count}")


#fourth join adv_line_df on purchase_key
order_level_4_df = order_level_3_df.join(adv_line_df, order_level_3_df.purchase_key == adv_line_df.purchase_key, how="inner").drop(adv_line_df['purchase_key']).drop(adv_line_df['quantity'])
display(order_level_4_df.limit(10))
print(f"order_level_4 count: {order_level_4_df.count()}")

#Check for duplicates in order_level_4_df
duplicate_rows_count = order_level_4_df.groupBy(order_level_4_df.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows in order_level_4_df: {duplicate_rows_count}")

#save order_level_4_df to volumes
order_level_4_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/order_level_4_df.csv")


In [0]:
%skip
#Now join the rest of the needed non-order level advantage tables to the joined order level tables

#Import sql functions for getting earliest and latest dates
from pyspark.sql import functions as F 
from pyspark.sql.functions import asc, desc

#Bring in order_level_4_df
order_level_4_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/order_level_4_df.csv"
order_level_4_df = spark.read.csv(order_level_4_path, header=True)
print(f"order_level_4_df count: {order_level_4_df.count()}")
order_level_4_cols = order_level_4_df.columns

#advantage_contract
adv_contract_csvpath ="/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_contract.csv"
adv_contract_df = spark.read.csv(adv_contract_csvpath, header=True)

#Join fields: "contract_key"- with advantage_line_item
adv_contract_count = adv_contract_df.count()
print(f"adv_contract_df count: {adv_contract_df.count()}")
print(f"adv_contract_df distinct gsam_cont_no values count: {adv_contract_df.select("gsam_cont_no").distinct().count()}")
print(f"adv_contract_df distinct contract_key values count: {adv_contract_df.select("contract_key").distinct().count()}")
ck_isnotnull_cont = adv_contract_df.filter(adv_contract_df.contract_key.isNotNull()).count()
print(f"count where contract_key is null: {adv_contract_count - ck_isnotnull_cont}")

#advantage_shopper
adv_shopper_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_shopper.csv"
adv_shopper_df = spark.read.csv(adv_shopper_csvpath, header=True)

#Join fields: "shopper_key"- with advantage_line_item
adv_shopper_count = adv_shopper_df.count()
print(f"adv_shopper_df count: {adv_shopper_df.count()}")
print(f"adv_shopper_df distinct agency_name values count: {adv_shopper_df.select("agency_name").distinct().count()}")
print(f"adv_shopper_df distinct shopper_key values count: {adv_shopper_df.select("shopper_key").distinct().count()}")
sk_isnotnull_shop = adv_shopper_df.filter(adv_shopper_df.shopper_key.isNotNull()).count()
print(f"count where shopper_key is null: {adv_shopper_count - sk_isnotnull_shop}")

#advantage_schedule
adv_schedule_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_schedule.csv"
adv_schedule_df = spark.read.csv(adv_schedule_csvpath, header=True)

#Join fields: "schedule_key" - with advantage_line_item
adv_schedule_count = adv_schedule_df.count()
print(f"adv_schedule_df count: {adv_schedule_df.count()}")
print(f"adv_schedule_df distinct sin values count: {adv_schedule_df.select("sin").distinct().count()}")
print(f"adv_schedule_df distinct schedule_key values count: {adv_schedule_df.select("schedule_key").distinct().count()}")
sk_isnotnull_sched = adv_schedule_df.filter(adv_schedule_df.schedule_key.isNotNull()).count()
print(f"count where schedule_key is null: {adv_schedule_count - sk_isnotnull_sched}")

#advantage_product
adv_product_csvpath = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_product.csv"
adv_product_df = spark.read.csv(adv_product_csvpath, header=True)

#Join fields: "mfr_part_number" should join with "mfr_part" from advantage_item_option, "product_key" - with advantage_line_item, "extract_day_key"- with advantage_purchase
adv_product_count = adv_product_df.count()
print(f"adv_product_df count: {adv_shopper_df.count()}")
print(f"adv_product_df distinct mfr_part_number values count: {adv_product_df.select("mfr_part_number").distinct().count()}")
print(f"adv_product_df distinct vend_part_number values count: {adv_product_df.select("vend_part_number").distinct().count()}")
print(f"adv_product_df distinct product_key values count: {adv_product_df.select("product_key").distinct().count()}")
pk_isnotnull_prod = adv_product_df.filter(adv_product_df.product_key.isNotNull()).count()
print(f"count where product_key is null: {adv_product_count - pk_isnotnull_prod}")


#first join: order_level_4_df and advantage_contract adv_contract_df on "contract_key"
#drop contract_key and gsam_cont_no
nonorder_join_5_df = order_level_4_df.join(adv_contract_df, order_level_4_df.contract_key == adv_contract_df.contract_key, how="inner").drop(adv_contract_df['contract_key']).drop(adv_contract_df['gsam_cont_no'])
display(nonorder_join_5_df.limit(10))
print(f"nonorder_join_5 count: {nonorder_join_5_df.count()}")

#Second join:  advantage_shopper adv_shopper_df on "shopper_key"
#drop shopper_key 
nonorder_join_6_df = nonorder_join_5_df.join(adv_shopper_df, nonorder_join_5_df.shopper_key == adv_shopper_df.shopper_key, how="left").drop(adv_shopper_df["shopper_key"])
display(nonorder_join_6_df.limit(10))
print(f"nonorder_join_6 count: {nonorder_join_6_df.count()}")

#third join:  advantage_schedule adv_schedule_df  on "schedule_key"
#drop schedule_key 
nonorder_join_7_df = nonorder_join_6_df.join(adv_schedule_df, nonorder_join_6_df.schedule_key == adv_schedule_df.schedule_key, how="left").drop(adv_schedule_df["schedule_key"])
display(nonorder_join_7_df.limit(10))
print(f"nonorder_join_7 count: {nonorder_join_7_df.count()}")

#fourth join with adv_product_df on product_key
#drop product_key  and extract_day_key
nonorder_join_8_df = nonorder_join_7_df.join(adv_product_df, nonorder_join_7_df.product_key == adv_product_df.product_key, how="left").drop(adv_product_df["product_key"]).drop(adv_product_df["extract_day_key"])
display(nonorder_join_8_df.limit(10))
print(f"nonorder_join_8 count: {nonorder_join_8_df.count()}")

#sort table by descencing order_date to make table in date order from most recent to last 
nonorder_join_8_df = nonorder_join_8_df.sort(nonorder_join_8_df.order_date.desc())

#save as csv to volumes
nonorder_join_8_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orders_allJoins.csv")

Redo joins without advantage_portal_order_assn.


In [0]:
%skip
from pyspark.sql import functions as F 

#Join on purchase numbers
#Bring in adv_orderstat_fss19_df 
adv_orderstat_fss19_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_orderstat_fss19_df.csv"
adv_orderstat_fss19_df = spark.read.csv(adv_orderstat_fss19_path, header=True)
print(f"Count of adv_orderstat_fss19_df: {adv_orderstat_fss19_df.count()}")

#Bring in adv_purchase_day
adv_purchase_day_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/adv_purchase_day.csv"
adv_purchase_day_df = spark.read.csv(adv_purchase_day_path, header=True)
print(f"Count of adv_purchase_day_df: {adv_purchase_day_df.count()}")

#join tables on purchase_number
join_orderPurchase_df = adv_orderstat_fss19_df.join(adv_purchase_day_df, adv_orderstat_fss19_df.purchase_number == adv_purchase_day_df.purchase_number, how="inner").drop(adv_purchase_day_df.purchase_number)
display(join_orderPurchase_df.limit(10))
print(f"join_orderPurchase_df count: {join_orderPurchase_df.count()}")

#save as csv to volumes
join_orderPurchase_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Data_Sets/DFsused/join_orderPurchase_df.csv")

Code for Calculated fields then join to Data Table

Delivery date comes from shippers API